# 04 - Expanding Window Experiment V2

Notebook-first, fully inspectable experiment for milk forecasting with:
- shared expanding-window splits,
- models: `XGBoost`, `LinearRegression`, `Prophet`, `ARIMA`,
- inverse-RMSE weighted ensemble,
- parent/child MLflow logging with stepped metric history,
- consolidated custom dumps for predictions, metrics, and final model artifacts.


In [ ]:
# =========================
# CONFIG (single block)
# =========================

import os
from pathlib import Path

ROOT = Path("..").resolve()

INPUT_CLEANED_MERGED_CSV = ROOT / "artifacts" / "data" / "cleaned_merged_with_ingredients.csv"
DATE_COLUMN_CANDIDATES = ["date", "Date", "bill_date", "created_at", "timestamp"]
TARGET_COLUMN_CANDIDATES = ["gallons", "total_milk_oz", "milk_oz"]

LOOKBACK_WINDOWS_DAYS = [45]
PREDICTION_WINDOW_DAYS = 30
EXPAND_STEP_DAYS = 30
INCLUDE_FORCED_FINAL_WINDOW = True

LAG_DAYS = [1, 2, 3, 7, 14]

RUN_OUTPUT_SUFFIX = os.getenv("RUN_OUTPUT_SUFFIX", "")
ENABLE_SMOKE_TEST = bool(int(os.getenv("SMOKE_TEST", "0")))
SMOKE_MAX_SPLITS = int(os.getenv("SMOKE_MAX_SPLITS", "2"))

OUTPUT_DIR = ROOT / "artifacts" / ("expanding_backtest_v2" if not RUN_OUTPUT_SUFFIX else f"expanding_backtest_v2_{RUN_OUTPUT_SUFFIX}")
PREDICTIONS_DIR = OUTPUT_DIR / "predictions"
METRICS_DIR = OUTPUT_DIR / "metrics"
MODELS_DIR = OUTPUT_DIR / "models"
SUMMARY_DIR = OUTPUT_DIR / "summary"

SELECTION_WEIGHT_AVG = 0.5

ENABLE_MLFLOW = True
TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI_OVERRIDE", "sqlite:///../mlruns_v2.db")
EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME_OVERRIDE", f"lookforward_{PREDICTION_WINDOW_DAYS}_{EXPAND_STEP_DAYS}_all_models")
COMPARISON_RUN_NAME = "model_comparison_v2"

SHOW_PLOTS = True
ENABLE_ENSEMBLE = False  # drop ensemble for now
TOP_MODELS_TO_PLOT = 6

print("Input:", INPUT_CLEANED_MERGED_CSV)
print("Lookbacks:", LOOKBACK_WINDOWS_DAYS)
print("Horizon:", PREDICTION_WINDOW_DAYS)
print("Step:", EXPAND_STEP_DAYS)
print("Smoke:", ENABLE_SMOKE_TEST, "max_splits:", SMOKE_MAX_SPLITS)
print("Output:", OUTPUT_DIR)
print("Ensemble enabled:", ENABLE_ENSEMBLE)







In [ ]:
import warnings
import re
import json
import math
import tempfile

import numpy as np
import pandas as pd
import cloudpickle
import matplotlib.pyplot as plt

missing = []

try:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.statespace.sarimax import SARIMAX
except Exception:
    ARIMA = None
    SARIMAX = None
    missing.append("statsmodels")

try:
    from prophet import Prophet
except Exception:
    Prophet = None
    missing.append("prophet")

try:
    from sklearn.linear_model import LinearRegression
except Exception:
    LinearRegression = None
    missing.append("scikit-learn")

try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor = None
    missing.append("xgboost")

if ENABLE_MLFLOW:
    try:
        import mlflow
    except Exception:
        mlflow = None
        missing.append("mlflow")
else:
    mlflow = None

if missing:
    raise RuntimeError(
        "Missing required package(s): "
        + ", ".join(sorted(set(missing)))
        + ". Install and rerun. Example: pip install scikit-learn xgboost prophet statsmodels mlflow"
    )

warnings.filterwarnings("ignore")
print("Dependencies OK")




In [ ]:
# =========================
# DATA LOAD + DAILY PREP
# =========================

def pick_column(columns, candidates):
    col_map = {c.lower(): c for c in columns}
    for c in candidates:
        if c.lower() in col_map:
            return col_map[c.lower()]
    return None

if not INPUT_CLEANED_MERGED_CSV.exists():
    raise FileNotFoundError(f"Missing {INPUT_CLEANED_MERGED_CSV}")

raw_df = pd.read_csv(INPUT_CLEANED_MERGED_CSV)

date_col = pick_column(raw_df.columns, DATE_COLUMN_CANDIDATES)
target_col = pick_column(raw_df.columns, TARGET_COLUMN_CANDIDATES)

if date_col is None:
    raise ValueError(f"No date column found. Tried: {DATE_COLUMN_CANDIDATES}")
if target_col is None:
    raise ValueError(f"No target column found. Tried: {TARGET_COLUMN_CANDIDATES}")

work_df = raw_df[[date_col, target_col]].copy()
work_df.columns = ["date", "target_raw"]
work_df["date"] = pd.to_datetime(work_df["date"], errors="coerce")
work_df["target_raw"] = pd.to_numeric(work_df["target_raw"], errors="coerce")
work_df = work_df.dropna(subset=["date", "target_raw"]).copy()

if target_col.lower() == "gallons":
    work_df["gallons"] = work_df["target_raw"]
else:
    work_df["gallons"] = work_df["target_raw"] / 128.0

work_df["date"] = work_df["date"].dt.normalize()
daily_df = (
    work_df.groupby("date", as_index=False)["gallons"]
    .sum()
    .sort_values("date")
    .reset_index(drop=True)
)

daily_df = daily_df.set_index("date").asfreq("D")
daily_df["gallons"] = daily_df["gallons"].fillna(0.0)
daily_df = daily_df.reset_index()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
derived_daily_path = OUTPUT_DIR / "derived_daily_from_cleaned_merged.csv"
daily_df.to_csv(derived_daily_path, index=False)

print("Source:", INPUT_CLEANED_MERGED_CSV)
print("Date column:", date_col)
print("Target column:", target_col)
print("Rows:", len(daily_df))
print("Date range:", daily_df["date"].min(), "->", daily_df["date"].max())
print("Mean gallons:", round(float(daily_df["gallons"].mean()), 3))
print("Saved derived daily:", derived_daily_path)
daily_df.head()



In [ ]:
# =========================
# UTILS: metrics, features, expanding splits
# =========================

def sanitize_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", name)


def safe_float(x):
    try:
        v = float(x)
        if np.isnan(v) or np.isinf(v):
            return float("nan")
        return v
    except Exception:
        return float("nan")


def metrics_dict(actual, pred):
    actual = np.asarray(actual, dtype=float)
    pred = np.asarray(pred, dtype=float)
    err = actual - pred

    rmse = float(np.sqrt(np.mean(err ** 2)))
    mae = float(np.mean(np.abs(err)))

    mask = actual != 0
    if np.any(mask):
        mape = float(np.mean(np.abs(err[mask] / actual[mask])) * 100.0)
    else:
        mape = float("nan")

    accuracy = float(100.0 - mape) if not np.isnan(mape) else float("nan")
    bias = float(np.mean(pred - actual))
    mean_actual = float(np.mean(actual)) if len(actual) else float("nan")
    error_pct = float((rmse / mean_actual) * 100.0) if (pd.notna(mean_actual) and mean_actual != 0) else float("nan")
    error_std = float(np.std(err))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "Accuracy": accuracy,
        "Bias": bias,
        "ErrorPct": error_pct,
        "ErrorStd": error_std,
    }


def _roll_mean(vals, w):
    if len(vals) < w:
        return float(np.mean(vals)) if len(vals) else 0.0
    return float(np.mean(vals[-w:]))


def _roll_std(vals, w):
    if len(vals) < w:
        return float(np.std(vals)) if len(vals) else 0.0
    return float(np.std(vals[-w:]))


def build_feature_row(history_vals, current_date, lag_days):
    eps = 1e-9
    row = {}

    for lag in lag_days:
        row[f"lag_{lag}"] = float(history_vals[-lag])

    mean7 = _roll_mean(history_vals, 7)
    mean14 = _roll_mean(history_vals, 14)

    row["mean_7"] = mean7
    row["mean_14"] = mean14
    row["std_7"] = _roll_std(history_vals, 7)
    row["std_14"] = _roll_std(history_vals, 14)
    row["mean_7_vs_14_diff"] = mean7 - mean14
    ratio = float(mean7 / (mean14 + eps))
    pct = float(((mean7 - mean14) / (abs(mean14) + eps)) * 100.0)
    row["mean_7_vs_14_ratio"] = float(np.clip(ratio, -10.0, 10.0))
    row["mean_7_vs_14_pct"] = float(np.clip(pct, -500.0, 500.0))

    cd = pd.Timestamp(current_date)
    row["day_of_week"] = int(cd.dayofweek)
    row["is_weekend"] = int(cd.dayofweek >= 5)
    row["month"] = int(cd.month)
    row["day_of_month"] = int(cd.day)
    row["week_of_year"] = int(cd.isocalendar().week)

    return row


def build_train_xy(train_dates, train_vals, lag_days):
    max_lag = max(lag_days)
    rows = []
    y = []
    y_dates = []

    for i in range(max_lag, len(train_vals)):
        feat = build_feature_row(train_vals[:i], train_dates[i], lag_days)
        rows.append(feat)
        y.append(float(train_vals[i]))
        y_dates.append(pd.Timestamp(train_dates[i]))

    if not rows:
        return pd.DataFrame(), np.array([]), np.array([])

    return pd.DataFrame(rows), np.asarray(y, dtype=float), np.asarray(y_dates)




def compute_recursive_pred_upper(train_vals):
    vals = np.asarray(train_vals, dtype=float)
    if len(vals) == 0:
        return 1.0
    p95 = float(np.quantile(vals, 0.95))
    vmax = float(np.max(vals))
    mean = float(np.mean(vals))
    std = float(np.std(vals))
    cands = [p95 * 2.0, vmax * 1.5, mean + 5.0 * std, 1.0]
    return float(max(cands))


def clip_feature_frame(x_row, feature_min_map=None, feature_max_map=None):
    if feature_min_map is None or feature_max_map is None:
        return x_row
    out = x_row.copy()
    for c in out.columns:
        lo = feature_min_map.get(c, None)
        hi = feature_max_map.get(c, None)
        if lo is None or hi is None:
            continue
        out[c] = np.clip(out[c], lo, hi)
    return out


def recursive_predict_feature_model(estimator, train_dates, train_vals, horizon_days, lag_days, feature_min_map=None, feature_max_map=None, pred_upper=None):
    history = list(np.asarray(train_vals, dtype=float))
    preds = []
    last_date = pd.Timestamp(train_dates[-1])

    for _ in range(horizon_days):
        next_date = last_date + pd.Timedelta(days=1)
        feat = build_feature_row(history, next_date, lag_days)
        x_next = pd.DataFrame([feat]).reindex(columns=FEATURE_COLUMNS, fill_value=0.0)
        x_next = clip_feature_frame(x_next, feature_min_map=feature_min_map, feature_max_map=feature_max_map)
        p = float(estimator.predict(x_next)[0])
        p = max(p, 0.0)
        if pred_upper is not None:
            p = min(p, float(pred_upper))
        preds.append(p)
        history.append(p)
        last_date = next_date

    return np.asarray(preds, dtype=float)


def generate_expanding_splits(n_rows, initial_train_days, horizon_days, step_days, include_forced_final=True):
    if n_rows <= initial_train_days + horizon_days:
        return []

    out = []
    train_end_idx = initial_train_days - 1

    while train_end_idx + horizon_days < n_rows:
        out.append({
            "train_start_idx": 0,
            "train_end_idx": train_end_idx,
            "test_start_idx": train_end_idx + 1,
            "test_end_idx": train_end_idx + horizon_days,
        })
        train_end_idx += step_days

    final_train_end = n_rows - horizon_days - 1
    if include_forced_final and final_train_end >= initial_train_days - 1:
        if not out or out[-1]["train_end_idx"] != final_train_end:
            out.append({
                "train_start_idx": 0,
                "train_end_idx": final_train_end,
                "test_start_idx": final_train_end + 1,
                "test_end_idx": final_train_end + horizon_days,
            })

    return out





In [ ]:
# =========================
# MODEL CATALOG
# =========================

model_catalog = {
    "XGBoost": {
        "family": "feature",
        "builder": lambda: XGBRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42,
            objective="reg:squarederror",
            n_jobs=-1,
        ),
        "hyperparams": {
            "n_estimators": 500,
            "learning_rate": 0.05,
            "max_depth": 5,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "random_state": 42,
            "objective": "reg:squarederror",
        },
    },
    "LinearRegression": {
        "family": "feature",
        "builder": lambda: LinearRegression(),
        "hyperparams": {"fit_intercept": True},
    },
    "ARIMA": {
        "family": "arima",
        "order": (2, 1, 2),
        "hyperparams": {"order": "(2,1,2)"},
    },
    "SARIMA": {
        "family": "sarima",
        "order": (2, 1, 2),
        "seasonal_order": (1, 1, 1, 7),
        "hyperparams": {"order": "(2,1,2)", "seasonal_order": "(1,1,1,7)"},
    },
    "Prophet": {
        "family": "prophet",
        "params": {
            "seasonality_mode": "multiplicative",
            "weekly_seasonality": True,
            "yearly_seasonality": False,
            "daily_seasonality": False,
            "changepoint_prior_scale": 0.05,
            "seasonality_prior_scale": 1.0,
        },
        "hyperparams": {
            "seasonality_mode": "multiplicative",
            "weekly_seasonality": True,
            "yearly_seasonality": False,
            "daily_seasonality": False,
            "changepoint_prior_scale": 0.05,
            "seasonality_prior_scale": 1.0,
        },
    },
}

FEATURE_COLUMNS = [f"lag_{x}" for x in LAG_DAYS] + [
    "mean_7",
    "mean_14",
    "std_7",
    "std_14",
    "mean_7_vs_14_diff",
    "mean_7_vs_14_ratio",
    "mean_7_vs_14_pct",
    "day_of_week",
    "is_weekend",
    "month",
    "day_of_month",
    "week_of_year",
]

print("Models:", sorted(model_catalog.keys()))
print("Feature columns:", len(FEATURE_COLUMNS))




In [ ]:
# =========================
# BACKTEST + CUSTOM DUMPS
# =========================

for d in [OUTPUT_DIR, PREDICTIONS_DIR, METRICS_DIR, MODELS_DIR, SUMMARY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

dates = daily_df["date"].to_numpy()
values = daily_df["gallons"].astype(float).to_numpy()

all_metrics_rows = []
all_prediction_rows = []
artifact_manifest = {}


def nan_metrics():
    return {
        "RMSE": np.nan,
        "MAE": np.nan,
        "MAPE": np.nan,
        "Accuracy": np.nan,
        "Bias": np.nan,
        "ErrorPct": np.nan,
        "ErrorStd": np.nan,
    }


for model_name, spec in model_catalog.items():
    model_metrics_rows = []
    model_prediction_rows = []
    latest_fitted_model = None
    latest_test_end = None
    split_idx_global = 0

    for lookback_days in LOOKBACK_WINDOWS_DAYS:
        splits = generate_expanding_splits(
            n_rows=len(daily_df),
            initial_train_days=lookback_days,
            horizon_days=PREDICTION_WINDOW_DAYS,
            step_days=EXPAND_STEP_DAYS,
            include_forced_final=INCLUDE_FORCED_FINAL_WINDOW,
        )

        if ENABLE_SMOKE_TEST:
            splits = splits[: min(SMOKE_MAX_SPLITS, len(splits))]

        for window_id, sp in enumerate(splits, start=1):
            split_idx_global += 1

            tr_s = sp["train_start_idx"]
            tr_e = sp["train_end_idx"]
            te_s = sp["test_start_idx"]
            te_e = sp["test_end_idx"]

            train_dates = dates[tr_s:tr_e + 1]
            train_vals = values[tr_s:tr_e + 1]
            test_dates = dates[te_s:te_e + 1]
            test_vals = values[te_s:te_e + 1]

            status = "ok"
            err_msg = ""
            fitted_obj = None
            preds = np.full(len(test_vals), np.nan)

            train_metrics = nan_metrics()
            train_eval_size = 0
            train_eval_start = str(pd.Timestamp(train_dates[0]).date())
            train_eval_end = str(pd.Timestamp(train_dates[-1]).date())

            try:
                fam = spec["family"]

                if fam == "feature":
                    est = spec["builder"]()
                    X_train, y_train, y_train_dates = build_train_xy(train_dates, train_vals, LAG_DAYS)
                    if len(X_train) < 10:
                        raise RuntimeError("not enough post-lag training rows")

                    est.fit(X_train, y_train)
                    train_pred = np.maximum(np.asarray(est.predict(X_train), dtype=float), 0.0)
                    train_metrics = metrics_dict(y_train, train_pred)

                    train_eval_size = int(len(y_train))
                    train_eval_start = str(pd.Timestamp(y_train_dates[0]).date())
                    train_eval_end = str(pd.Timestamp(y_train_dates[-1]).date())

                    feat_min = {c: float(v) for c, v in X_train.min().to_dict().items()}
                    feat_max = {c: float(v) for c, v in X_train.max().to_dict().items()}
                    pred_upper = compute_recursive_pred_upper(train_vals)

                    preds = recursive_predict_feature_model(
                        estimator=est,
                        train_dates=train_dates,
                        train_vals=train_vals,
                        horizon_days=len(test_vals),
                        lag_days=LAG_DAYS,
                        feature_min_map=feat_min,
                        feature_max_map=feat_max,
                        pred_upper=pred_upper,
                    )
                    fitted_obj = est

                elif fam == "arima":
                    fit = ARIMA(train_vals, order=spec["order"]).fit()
                    preds = np.maximum(np.asarray(fit.forecast(steps=len(test_vals)), dtype=float), 0.0)

                    fitted_train = np.asarray(fit.fittedvalues, dtype=float)
                    actual_train = np.asarray(train_vals[-len(fitted_train):], dtype=float)
                    fitted_train = np.maximum(fitted_train, 0.0)
                    mask = np.isfinite(actual_train) & np.isfinite(fitted_train)
                    if int(mask.sum()) >= 5:
                        train_metrics = metrics_dict(actual_train[mask], fitted_train[mask])
                    train_eval_size = int(mask.sum())

                    fitted_obj = fit

                elif fam == "sarima":
                    fit = SARIMAX(
                        train_vals,
                        order=spec["order"],
                        seasonal_order=spec["seasonal_order"],
                        enforce_stationarity=False,
                        enforce_invertibility=False,
                    ).fit(disp=False)

                    preds = np.maximum(np.asarray(fit.get_forecast(steps=len(test_vals)).predicted_mean, dtype=float), 0.0)

                    fitted_train = np.asarray(fit.fittedvalues, dtype=float)
                    actual_train = np.asarray(train_vals[-len(fitted_train):], dtype=float)
                    fitted_train = np.maximum(fitted_train, 0.0)
                    mask = np.isfinite(actual_train) & np.isfinite(fitted_train)
                    if int(mask.sum()) >= 5:
                        train_metrics = metrics_dict(actual_train[mask], fitted_train[mask])
                    train_eval_size = int(mask.sum())

                    fitted_obj = fit

                elif fam == "prophet":
                    tdf = pd.DataFrame({"ds": pd.to_datetime(train_dates), "y": np.asarray(train_vals, dtype=float)})
                    pm = Prophet(**spec["params"])
                    pm.fit(tdf)

                    in_sample = pm.predict(tdf[["ds"]])["yhat"].to_numpy(dtype=float)
                    in_sample = np.maximum(in_sample, 0.0)
                    train_metrics = metrics_dict(np.asarray(train_vals, dtype=float), in_sample)
                    train_eval_size = int(len(train_vals))

                    fut = pm.make_future_dataframe(periods=len(test_vals), freq="D")
                    fc = pm.predict(fut).tail(len(test_vals))
                    preds = np.maximum(np.asarray(fc["yhat"].to_numpy(), dtype=float), 0.0)
                    fitted_obj = pm

                else:
                    raise RuntimeError(f"unknown model family: {fam}")

            except Exception as e:
                status = "error"
                err_msg = f"{type(e).__name__}: {e}"
                preds = np.full(len(test_vals), np.nan)
                train_metrics = nan_metrics()

            if np.isnan(preds).any():
                test_metrics = nan_metrics()
            else:
                test_metrics = metrics_dict(test_vals, preds)

            if pd.notna(test_metrics["RMSE"]) and pd.notna(train_metrics["RMSE"]):
                gap_rmse = float(test_metrics["RMSE"] - train_metrics["RMSE"])
            else:
                gap_rmse = np.nan

            row = {
                "model": model_name,
                "lookback_days": int(lookback_days),
                "window_id": int(window_id),
                "split_idx_global": int(split_idx_global),
                "status": status,
                "error_message": err_msg,
                "train_start": str(pd.Timestamp(train_dates[0]).date()),
                "train_end": str(pd.Timestamp(train_dates[-1]).date()),
                "test_start": str(pd.Timestamp(test_dates[0]).date()),
                "test_end": str(pd.Timestamp(test_dates[-1]).date()),
                "n_train": int(len(train_vals)),
                "n_test": int(len(test_vals)),
                "n_train_eval": int(train_eval_size),
                "train_eval_start": str(train_eval_start),
                "train_eval_end": str(train_eval_end),
                "generalization_gap_RMSE": gap_rmse,
                "model_hyperparams_json": json.dumps(spec.get("hyperparams", {}), sort_keys=True),
                "feature_columns_json": json.dumps(FEATURE_COLUMNS if spec.get("family") == "feature" else []),
            }
            row.update(test_metrics)
            row.update({f"train_{k}": v for k, v in train_metrics.items()})
            model_metrics_rows.append(row)

            for dte, act, prd in zip(test_dates, test_vals, preds):
                err = act - prd if pd.notna(prd) else np.nan
                ape = (abs(err) / abs(act) * 100.0) if (pd.notna(prd) and act != 0) else np.nan
                model_prediction_rows.append(
                    {
                        "model": model_name,
                        "lookback_days": int(lookback_days),
                        "window_id": int(window_id),
                        "split_idx_global": int(split_idx_global),
                        "train_start": row["train_start"],
                        "train_end": row["train_end"],
                        "test_start": row["test_start"],
                        "test_end": row["test_end"],
                        "date": str(pd.Timestamp(dte).date()),
                        "actual": float(act),
                        "prediction": float(prd) if pd.notna(prd) else np.nan,
                        "error": float(err) if pd.notna(err) else np.nan,
                        "abs_error": float(abs(err)) if pd.notna(err) else np.nan,
                        "ape": float(ape) if pd.notna(ape) else np.nan,
                    }
                )

            if status == "ok":
                cur_end = pd.Timestamp(test_dates[-1])
                if latest_test_end is None or cur_end > latest_test_end:
                    latest_test_end = cur_end
                    latest_fitted_model = fitted_obj

    model_metrics_df = pd.DataFrame(model_metrics_rows)
    model_predictions_df = pd.DataFrame(model_prediction_rows)

    safe = sanitize_name(model_name)
    model_dir = MODELS_DIR / safe
    model_dir.mkdir(parents=True, exist_ok=True)

    metrics_path = METRICS_DIR / f"{safe}_window_metrics.csv"
    preds_path = PREDICTIONS_DIR / f"{safe}_predictions_all_splits.csv"
    model_path = model_dir / "final_model.pkl"
    model_config_path = model_dir / "model_config.json"
    feature_schema_path = model_dir / "feature_schema.json"

    model_metrics_df.to_csv(metrics_path, index=False)
    model_predictions_df.to_csv(preds_path, index=False)

    if latest_fitted_model is not None:
        with open(model_path, "wb") as f:
            cloudpickle.dump(latest_fitted_model, f)
    else:
        model_path.write_text("No successful fitted model available", encoding="utf-8")

    model_config = {
        "model": model_name,
        "family": spec.get("family"),
        "hyperparams": spec.get("hyperparams", {}),
        "latest_test_end": str(latest_test_end.date()) if latest_test_end is not None else None,
        "lookbacks": LOOKBACK_WINDOWS_DAYS,
        "horizon_days": PREDICTION_WINDOW_DAYS,
        "step_days": EXPAND_STEP_DAYS,
    }
    model_config_path.write_text(json.dumps(model_config, indent=2), encoding="utf-8")

    feature_schema = {
        "model": model_name,
        "target_column": "gallons",
        "feature_columns": FEATURE_COLUMNS if spec.get("family") == "feature" else [],
    }
    feature_schema_path.write_text(json.dumps(feature_schema, indent=2), encoding="utf-8")

    artifact_manifest[model_name] = {
        "metrics_csv": str(metrics_path),
        "predictions_csv": str(preds_path),
        "final_model": str(model_path),
        "model_config_json": str(model_config_path),
        "feature_schema_json": str(feature_schema_path),
        "weights_csv": None,
        "hyperparams": spec.get("hyperparams", {}),
        "feature_columns": FEATURE_COLUMNS if spec.get("family") == "feature" else [],
    }

    all_metrics_rows.extend(model_metrics_rows)
    all_prediction_rows.extend(model_prediction_rows)
    print(f"Done model: {model_name} | windows: {len(model_metrics_rows)}")


all_metrics_df = pd.DataFrame(all_metrics_rows)
all_predictions_df = pd.DataFrame(all_prediction_rows)

all_metrics_path = METRICS_DIR / "all_models_metrics.csv"
all_preds_path = PREDICTIONS_DIR / "all_models_predictions.csv"
all_metrics_df.to_csv(all_metrics_path, index=False)
all_predictions_df.to_csv(all_preds_path, index=False)

print("Saved:", all_metrics_path)
print("Saved:", all_preds_path)






In [ ]:
# =========================
# ENSEMBLE (inverse historical RMSE)
# =========================

if ENABLE_ENSEMBLE:
    # =========================
    # ENSEMBLE (inverse historical RMSE)
    # =========================

    ensemble_name = "Ensemble_InverseRMSE"
    base_models = ["XGBoost", "LinearRegression", "Prophet", "ARIMA"]

    base_ok_metrics = all_metrics_df[(all_metrics_df["status"] == "ok") & (all_metrics_df["model"].isin(base_models))].copy()
    all_split_ids = sorted(all_metrics_df["split_idx_global"].dropna().astype(int).unique().tolist())

    ensemble_metrics_rows = []
    ensemble_prediction_rows = []
    ensemble_weight_rows = []

    for split_idx in all_split_ids:
        available = []
        pred_blocks = []
        for m in base_models:
            pp = all_predictions_df[(all_predictions_df["model"] == m) & (all_predictions_df["split_idx_global"] == split_idx)].copy()
            if pp.empty or pp["prediction"].notna().sum() == 0:
                continue
            keep = pp[["date", "actual", "prediction"]].copy().rename(columns={"prediction": f"pred_{m}"})
            pred_blocks.append(keep)
            available.append(m)

        if not pred_blocks:
            continue

        merged = pred_blocks[0][["date", "actual", f"pred_{available[0]}"]].copy()
        for i in range(1, len(pred_blocks)):
            merged = merged.merge(pred_blocks[i][["date", f"pred_{available[i]}"]], on="date", how="inner")

        hist_rmse = {}
        for m in available:
            hist = base_ok_metrics[(base_ok_metrics["model"] == m) & (base_ok_metrics["split_idx_global"] < split_idx)]["RMSE"].dropna()
            if len(hist) > 0:
                hist_rmse[m] = float(hist.mean())

        if len(hist_rmse) == len(available):
            inv = {m: 1.0 / max(hist_rmse[m], 1e-8) for m in available}
            den = sum(inv.values())
            weights = {m: inv[m] / den for m in available}
            weight_source = "inverse_historical_rmse"
        else:
            w = 1.0 / float(len(available))
            weights = {m: w for m in available}
            weight_source = "equal_fallback"

        merged["prediction"] = 0.0
        for m in available:
            merged["prediction"] += weights[m] * merged[f"pred_{m}"]

        tm = all_metrics_df[all_metrics_df["split_idx_global"] == split_idx].iloc[0]
        test_metrics = metrics_dict(merged["actual"].to_numpy(dtype=float), merged["prediction"].to_numpy(dtype=float))

        train_proxy = {}
        for mk in ["RMSE", "MAE", "MAPE", "Accuracy", "Bias", "ErrorPct", "ErrorStd"]:
            pairs = []
            for m in available:
                r = base_ok_metrics[(base_ok_metrics["model"] == m) & (base_ok_metrics["split_idx_global"] == split_idx)]
                if r.empty:
                    continue
                v = safe_float(r.iloc[0].get(f"train_{mk}", np.nan))
                if pd.notna(v):
                    pairs.append((weights[m], v))
            if pairs:
                train_proxy[mk] = float(sum(w * v for w, v in pairs) / sum(w for w, _ in pairs))
            else:
                train_proxy[mk] = np.nan

        gap_rmse = test_metrics["RMSE"] - train_proxy["RMSE"] if pd.notna(test_metrics["RMSE"]) and pd.notna(train_proxy["RMSE"]) else np.nan

        row = {
            "model": ensemble_name,
            "lookback_days": int(tm["lookback_days"]),
            "window_id": int(tm["window_id"]),
            "split_idx_global": int(split_idx),
            "status": "ok",
            "error_message": "",
            "train_start": str(tm["train_start"]),
            "train_end": str(tm["train_end"]),
            "test_start": str(tm["test_start"]),
            "test_end": str(tm["test_end"]),
            "n_train": int(tm["n_train"]),
            "n_test": int(tm["n_test"]),
            "n_train_eval": int(tm["n_train"]),
            "train_eval_start": str(tm["train_start"]),
            "train_eval_end": str(tm["train_end"]),
            "generalization_gap_RMSE": safe_float(gap_rmse),
            "model_hyperparams_json": json.dumps({"weighting": "inverse_historical_rmse", "members": available}),
            "feature_columns_json": json.dumps([]),
        }
        row.update(test_metrics)
        row.update({f"train_{k}": train_proxy[k] for k in train_proxy})
        ensemble_metrics_rows.append(row)

        for _, rr in merged.iterrows():
            err = rr["actual"] - rr["prediction"]
            ape = (abs(err) / abs(rr["actual"]) * 100.0) if rr["actual"] != 0 else np.nan
            ensemble_prediction_rows.append(
                {
                    "model": ensemble_name,
                    "lookback_days": int(tm["lookback_days"]),
                    "window_id": int(tm["window_id"]),
                    "split_idx_global": int(split_idx),
                    "train_start": str(tm["train_start"]),
                    "train_end": str(tm["train_end"]),
                    "test_start": str(tm["test_start"]),
                    "test_end": str(tm["test_end"]),
                    "date": str(rr["date"]),
                    "actual": float(rr["actual"]),
                    "prediction": float(rr["prediction"]),
                    "error": float(err),
                    "abs_error": float(abs(err)),
                    "ape": float(ape) if pd.notna(ape) else np.nan,
                }
            )

        for m in available:
            ensemble_weight_rows.append(
                {
                    "split_idx_global": int(split_idx),
                    "member_model": m,
                    "weight": float(weights[m]),
                    "weight_source": weight_source,
                }
            )

    ensemble_metrics_df = pd.DataFrame(ensemble_metrics_rows)
    ensemble_predictions_df = pd.DataFrame(ensemble_prediction_rows)
    ensemble_weights_df = pd.DataFrame(ensemble_weight_rows)

    if not ensemble_metrics_df.empty:
        all_metrics_df = pd.concat([all_metrics_df, ensemble_metrics_df], ignore_index=True)
    if not ensemble_predictions_df.empty:
        all_predictions_df = pd.concat([all_predictions_df, ensemble_predictions_df], ignore_index=True)

    ensemble_safe = sanitize_name(ensemble_name)
    ensemble_metrics_path = METRICS_DIR / f"{ensemble_safe}_window_metrics.csv"
    ensemble_preds_path = PREDICTIONS_DIR / f"{ensemble_safe}_predictions_all_splits.csv"
    ensemble_weights_path = SUMMARY_DIR / f"{ensemble_safe}_weights.csv"

    ensemble_metrics_df.to_csv(ensemble_metrics_path, index=False)
    ensemble_predictions_df.to_csv(ensemble_preds_path, index=False)
    ensemble_weights_df.to_csv(ensemble_weights_path, index=False)

    ensemble_dir = MODELS_DIR / ensemble_safe
    ensemble_dir.mkdir(parents=True, exist_ok=True)
    ensemble_model_path = ensemble_dir / "final_model.json"
    ensemble_model_path.write_text(json.dumps({"type": "weighted_average", "weights": ensemble_weight_rows}, indent=2), encoding="utf-8")
    model_config_path = ensemble_dir / "model_config.json"
    model_config_path.write_text(json.dumps({"model": ensemble_name, "weighting": "inverse_historical_rmse", "members": base_models}, indent=2), encoding="utf-8")
    feature_schema_path = ensemble_dir / "feature_schema.json"
    feature_schema_path.write_text(json.dumps({"model": ensemble_name, "feature_columns": []}, indent=2), encoding="utf-8")

    artifact_manifest[ensemble_name] = {
        "metrics_csv": str(ensemble_metrics_path),
        "predictions_csv": str(ensemble_preds_path),
        "final_model": str(ensemble_model_path),
        "model_config_json": str(model_config_path),
        "feature_schema_json": str(feature_schema_path),
        "weights_csv": str(ensemble_weights_path),
        "hyperparams": {"weighting": "inverse_historical_rmse", "members": base_models},
        "feature_columns": [],
    }

    all_metrics_df.to_csv(all_metrics_path, index=False)
    all_predictions_df.to_csv(all_preds_path, index=False)

    print("Ensemble done")
    print("Ensemble rows:", len(ensemble_metrics_df), len(ensemble_predictions_df))

else:
    print("Ensemble skipped (ENABLE_ENSEMBLE=False)")


In [ ]:
# =========================
# SUMMARY: latest vs average over time (base models unless ensemble enabled)
# =========================

ok_metrics = all_metrics_df[all_metrics_df["status"] == "ok"].copy()
if ok_metrics.empty:
    raise RuntimeError("No successful runs found")

ok_metrics["test_end_dt"] = pd.to_datetime(ok_metrics["test_end"])

metric_cols = ["RMSE", "MAE", "MAPE", "Accuracy", "Bias", "ErrorPct", "ErrorStd"]
for c in metric_cols:
    ok_metrics[c] = pd.to_numeric(ok_metrics[c], errors="coerce")

avg_df = ok_metrics.groupby("model", as_index=False)[metric_cols].mean()
avg_df = avg_df.rename(columns={c: f"avg_{c}" for c in metric_cols})

idx_latest = ok_metrics.sort_values("test_end_dt").groupby("model")["test_end_dt"].idxmax()
latest_df = ok_metrics.loc[idx_latest, ["model", "lookback_days", "window_id", "split_idx_global", "test_end"] + metric_cols].copy()
latest_df = latest_df.rename(columns={c: f"latest_{c}" for c in metric_cols})

trend_rows = []
for model_name, g in ok_metrics.sort_values("split_idx_global").groupby("model"):
    gg = g.reset_index(drop=True)
    if len(gg) >= 2:
        x = gg["split_idx_global"].to_numpy(dtype=float)
        y = gg["RMSE"].to_numpy(dtype=float)
        slope = float(np.polyfit(x, y, deg=1)[0])
        delta = float(y[-1] - y[0])
    else:
        slope = np.nan
        delta = np.nan
    trend_rows.append({"model": model_name, "rmse_trend_slope": slope, "rmse_trend_delta": delta})
trend_df = pd.DataFrame(trend_rows)

summary_df = avg_df.merge(latest_df, on="model", how="left").merge(trend_df, on="model", how="left")
summary_df["latest_rmse_rank"] = summary_df["latest_RMSE"].rank(method="dense")
summary_df["avg_rmse_rank"] = summary_df["avg_RMSE"].rank(method="dense")
summary_df["final_score"] = summary_df["latest_rmse_rank"] + SELECTION_WEIGHT_AVG * summary_df["avg_rmse_rank"]
summary_df = summary_df.sort_values(["final_score", "latest_RMSE", "avg_RMSE"]).reset_index(drop=True)

best_model = summary_df.iloc[0]["model"]

summary_path = SUMMARY_DIR / "model_comparison_summary.csv"
summary_df.to_csv(summary_path, index=False)

latest_vs_avg_path = SUMMARY_DIR / "latest_vs_avg_rmse.csv"
summary_df[["model", "latest_RMSE", "avg_RMSE", "rmse_trend_slope", "rmse_trend_delta", "final_score"]].to_csv(latest_vs_avg_path, index=False)

print("Best model candidate:", best_model)
print("Saved:", summary_path)
print("Saved:", latest_vs_avg_path)
summary_df.head(20)




In [ ]:
# =========================
# PLOTS + FINAL VIEW
# =========================

ok_metrics = all_metrics_df[all_metrics_df["status"] == "ok"].copy()
ok_metrics["test_end_dt"] = pd.to_datetime(ok_metrics["test_end"])
comparison_plot_path = SUMMARY_DIR / "all_models_comparison.png"

if SHOW_PLOTS and not ok_metrics.empty:
    top_models = summary_df["model"].head(TOP_MODELS_TO_PLOT).tolist()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for m in top_models:
        g = ok_metrics[ok_metrics["model"] == m].sort_values("split_idx_global")
        axes[0].plot(g["split_idx_global"], g["RMSE"], marker="o", label=m)
    axes[0].set_title("RMSE Over Splits")
    axes[0].set_xlabel("Split Index")
    axes[0].set_ylabel("RMSE")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(fontsize=8)

    show_df = summary_df[["model", "latest_RMSE", "avg_RMSE"]].head(TOP_MODELS_TO_PLOT).copy()
    x = np.arange(len(show_df))
    w = 0.38
    axes[1].bar(x - w / 2, show_df["latest_RMSE"], width=w, label="Latest RMSE")
    axes[1].bar(x + w / 2, show_df["avg_RMSE"], width=w, label="Avg RMSE")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(show_df["model"], rotation=30, ha="right")
    axes[1].set_title("Latest vs Average RMSE")
    axes[1].set_ylabel("RMSE")
    axes[1].grid(True, axis="y", alpha=0.3)
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(comparison_plot_path, dpi=150)
    plt.show()

print("Deployment candidate model:", best_model)
print("Saved chart:", comparison_plot_path)
summary_df[["model", "latest_RMSE", "avg_RMSE", "rmse_trend_slope", "rmse_trend_delta", "final_score"]].head(20)



In [ ]:
# =========================
# MLFLOW LOGGING
# parent per model, nested per window
# plus explicit final expanded nested run
# =========================

if ENABLE_MLFLOW and mlflow is not None:
    import tempfile

    mlflow.set_tracking_uri(TRACKING_URI)
    exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    experiment_id = exp.experiment_id if exp is not None else mlflow.create_experiment(EXPERIMENT_NAME)

    def clean_metrics(d):
        out = {}
        for k, v in d.items():
            try:
                fv = float(v)
            except Exception:
                continue
            if np.isnan(fv) or np.isinf(fv):
                continue
            out[k] = fv
        return out

    ok_metrics = all_metrics_df[all_metrics_df["status"] == "ok"].copy()
    ok_metrics["test_end_dt"] = pd.to_datetime(ok_metrics["test_end"])

    for model_name, g in ok_metrics.groupby("model"):
        g = g.sort_values(["split_idx_global", "test_end_dt", "lookback_days", "window_id"]).reset_index(drop=True)
        latest = g.sort_values("test_end_dt").iloc[-1].to_dict()
        avg = g[["RMSE", "MAE", "MAPE", "Accuracy", "Bias", "ErrorPct", "ErrorStd"]].mean().to_dict()

        with mlflow.start_run(experiment_id=experiment_id, run_name=f"{sanitize_name(model_name)}_parent"):
            # parent run identity + setup
            mlflow.set_tags({
                "run_type": "parent",
                "model_name": model_name,
                "cv_scheme": "expanding_window",
                "window_scheme": f"lookbacks_{'-'.join(str(x) for x in LOOKBACK_WINDOWS_DAYS)}_h{PREDICTION_WINDOW_DAYS}_s{EXPAND_STEP_DAYS}",
                "feature_version": "lag_roll_v2",
                "input_source": str(INPUT_CLEANED_MERGED_CSV),
            })

            mlflow.log_param("model", model_name)
            mlflow.log_param("lookbacks", ",".join(str(x) for x in LOOKBACK_WINDOWS_DAYS))
            mlflow.log_param("input_csv", str(INPUT_CLEANED_MERGED_CSV))
            mlflow.log_param("horizon_days", PREDICTION_WINDOW_DAYS)
            mlflow.log_param("step_days", EXPAND_STEP_DAYS)
            mlflow.log_param("lag_days", ",".join(str(x) for x in LAG_DAYS))

            model_meta = artifact_manifest.get(model_name, {})
            feature_cols = model_meta.get("feature_columns", [])
            hyperparams = model_meta.get("hyperparams", {})
            mlflow.log_param("feature_columns", ",".join(feature_cols))
            mlflow.log_param("feature_count", len(feature_cols))
            for hk, hv in hyperparams.items():
                mlflow.log_param(f"hp_{hk}", hv if isinstance(hv, (str, int, float, bool)) else str(hv))

            # parent aggregate metrics
            parent_metrics = {f"avg_{k}": v for k, v in avg.items()}
            parent_metrics.update({f"latest_{k}": latest[k] for k in ["RMSE", "MAE", "MAPE", "Accuracy", "Bias", "ErrorPct", "ErrorStd"]})
            first_rmse = safe_float(g.iloc[0]["RMSE"])
            last_rmse = safe_float(g.iloc[-1]["RMSE"])
            if pd.notna(first_rmse) and pd.notna(last_rmse):
                parent_metrics["rmse_trend_delta"] = float(last_rmse - first_rmse)
            if len(g) >= 2:
                x = g["split_idx_global"].to_numpy(dtype=float)
                y = g["RMSE"].to_numpy(dtype=float)
                parent_metrics["rmse_trend_slope"] = float(np.polyfit(x, y, deg=1)[0])
            mlflow.log_metrics(clean_metrics(parent_metrics))

            # stepped metric history on parent
            for _, r in g.iterrows():
                step_idx = int(r["split_idx_global"])

                mlflow.log_metric("test_rmse", float(r["RMSE"]), step=step_idx)
                mlflow.log_metric("test_mae", float(r["MAE"]), step=step_idx)
                if pd.notna(r["MAPE"]):
                    mlflow.log_metric("test_mape", float(r["MAPE"]), step=step_idx)
                if pd.notna(r["Accuracy"]):
                    mlflow.log_metric("test_accuracy", float(r["Accuracy"]), step=step_idx)
                if pd.notna(r["Bias"]):
                    mlflow.log_metric("test_bias", float(r["Bias"]), step=step_idx)
                if pd.notna(r["ErrorPct"]):
                    mlflow.log_metric("test_error_pct", float(r["ErrorPct"]), step=step_idx)
                if pd.notna(r["ErrorStd"]):
                    mlflow.log_metric("test_error_std", float(r["ErrorStd"]), step=step_idx)

                if pd.notna(r["train_RMSE"]):
                    mlflow.log_metric("train_rmse", float(r["train_RMSE"]), step=step_idx)
                if pd.notna(r["train_MAE"]):
                    mlflow.log_metric("train_mae", float(r["train_MAE"]), step=step_idx)
                if pd.notna(r["train_MAPE"]):
                    mlflow.log_metric("train_mape", float(r["train_MAPE"]), step=step_idx)
                if pd.notna(r["train_Accuracy"]):
                    mlflow.log_metric("train_accuracy", float(r["train_Accuracy"]), step=step_idx)
                if pd.notna(r["train_Bias"]):
                    mlflow.log_metric("train_bias", float(r["train_Bias"]), step=step_idx)
                if pd.notna(r["train_ErrorPct"]):
                    mlflow.log_metric("train_error_pct", float(r["train_ErrorPct"]), step=step_idx)
                if pd.notna(r["train_ErrorStd"]):
                    mlflow.log_metric("train_error_std", float(r["train_ErrorStd"]), step=step_idx)

                if pd.notna(r["generalization_gap_RMSE"]):
                    mlflow.log_metric("generalization_gap_rmse", float(r["generalization_gap_RMSE"]), step=step_idx)

                mlflow.log_metric("train_window_size", float(r["n_train"]), step=step_idx)
                mlflow.log_metric("test_window_size", float(r["n_test"]), step=step_idx)
                mlflow.log_metric("train_end_ordinal", float(pd.Timestamp(r["train_end"]).toordinal()), step=step_idx)
                mlflow.log_metric("test_end_ordinal", float(pd.Timestamp(r["test_end"]).toordinal()), step=step_idx)

            # explicit final expanded nested run
            with mlflow.start_run(experiment_id=experiment_id, run_name=f"{model_name}_final_expanded", nested=True):
                mlflow.set_tags({"run_type": "child", "run_role": "final_expanded", "model_name": model_name})
                mlflow.log_param("lookback_days", int(latest["lookback_days"]))
                mlflow.log_param("window_id", int(latest["window_id"]))
                mlflow.log_param("split_idx_global", int(latest["split_idx_global"]))
                mlflow.log_param("train_start", str(latest["train_start"]))
                mlflow.log_param("train_end", str(latest["train_end"]))
                mlflow.log_param("test_start", str(latest["test_start"]))
                mlflow.log_param("test_end", str(latest["test_end"]))
                mlflow.log_metrics(
                    clean_metrics(
                        {
                            "test_rmse": latest["RMSE"],
                            "test_mae": latest["MAE"],
                            "test_mape": latest["MAPE"],
                            "test_accuracy": latest["Accuracy"],
                            "test_bias": latest["Bias"],
                            "test_error_pct": latest["ErrorPct"],
                            "test_error_std": latest["ErrorStd"],
                            "train_rmse": latest["train_RMSE"],
                            "train_mae": latest["train_MAE"],
                            "train_mape": latest["train_MAPE"],
                            "train_accuracy": latest["train_Accuracy"],
                            "train_bias": latest["train_Bias"],
                            "train_error_pct": latest["train_ErrorPct"],
                            "train_error_std": latest["train_ErrorStd"],
                            "generalization_gap_rmse": latest["generalization_gap_RMSE"],
                        }
                    )
                )

            # child run per split with split-level artifacts
            for _, r in g.iterrows():
                split_idx = int(r["split_idx_global"])
                run_name = f"split_{split_idx:03d}_lb{int(r['lookback_days'])}_w{int(r['window_id'])}"

                with mlflow.start_run(experiment_id=experiment_id, run_name=run_name, nested=True):
                    mlflow.set_tags({
                        "run_type": "child",
                        "run_role": "window",
                        "model_name": model_name,
                    })

                    mlflow.log_params(
                        {
                            "model": model_name,
                            "split_idx": split_idx,
                            "lookback_days": int(r["lookback_days"]),
                            "window_id": int(r["window_id"]),
                            "train_start": str(r["train_start"]),
                            "train_end": str(r["train_end"]),
                            "test_start": str(r["test_start"]),
                            "test_end": str(r["test_end"]),
                            "train_size": int(r["n_train"]),
                            "test_size": int(r["n_test"]),
                            "feature_columns": ",".join(feature_cols),
                        }
                    )
                    for hk, hv in hyperparams.items():
                        mlflow.log_param(f"hp_{hk}", hv if isinstance(hv, (str, int, float, bool)) else str(hv))

                    mlflow.log_metrics(
                        clean_metrics(
                            {
                                "train_rmse": r["train_RMSE"],
                                "train_mae": r["train_MAE"],
                                "train_mape": r["train_MAPE"],
                                "train_accuracy": r["train_Accuracy"],
                                "train_bias": r["train_Bias"],
                                "train_error_pct": r["train_ErrorPct"],
                                "train_error_std": r["train_ErrorStd"],
                                "test_rmse": r["RMSE"],
                                "test_mae": r["MAE"],
                                "test_mape": r["MAPE"],
                                "test_accuracy": r["Accuracy"],
                                "test_bias": r["Bias"],
                                "test_error_pct": r["ErrorPct"],
                                "test_error_std": r["ErrorStd"],
                                "generalization_gap_rmse": r["generalization_gap_RMSE"],
                            }
                        )
                    )

                    child_preds = all_predictions_df[
                        (all_predictions_df["model"] == model_name)
                        & (all_predictions_df["split_idx_global"] == split_idx)
                    ].copy()

                    with tempfile.TemporaryDirectory(prefix="mlflow_split_") as td:
                        td_path = Path(td)
                        preds_csv = td_path / "predictions.csv"
                        residuals_csv = td_path / "residuals.csv"
                        child_preds.to_csv(preds_csv, index=False)

                        res_df = child_preds[["date", "actual", "prediction", "error", "abs_error", "ape"]].copy()
                        res_df.to_csv(residuals_csv, index=False)

                        mlflow.log_artifact(str(preds_csv), artifact_path="split_artifacts")
                        mlflow.log_artifact(str(residuals_csv), artifact_path="split_artifacts")

            # parent artifacts
            paths = artifact_manifest.get(model_name, {})
            for key in ["metrics_csv", "predictions_csv", "final_model", "model_config_json", "feature_schema_json", "weights_csv"]:
                p = paths.get(key)
                if p and Path(p).exists():
                    mlflow.log_artifact(str(p), artifact_path=sanitize_name(model_name))

            if Path(summary_path).exists():
                mlflow.log_artifact(str(summary_path), artifact_path="summary")
            if Path(latest_vs_avg_path).exists():
                mlflow.log_artifact(str(latest_vs_avg_path), artifact_path="summary")

    with mlflow.start_run(experiment_id=experiment_id, run_name=COMPARISON_RUN_NAME):
        mlflow.set_tags({
            "run_type": "comparison",
            "cv_scheme": "expanding_window",
            "input_source": str(INPUT_CLEANED_MERGED_CSV),
        })
        mlflow.log_params({
            "initial_train_days": LOOKBACK_WINDOWS_DAYS[0],
            "horizon_days": PREDICTION_WINDOW_DAYS,
            "step_days": EXPAND_STEP_DAYS,
            "model_count": int(summary_df["model"].nunique()),
            "best_model": str(best_model),
        })
        if not summary_df.empty:
            mlflow.log_metric("best_final_score", float(summary_df.iloc[0]["final_score"]))
            mlflow.log_metric("best_latest_rmse", float(summary_df.iloc[0]["latest_RMSE"]))
            for _, rr in summary_df.iterrows():
                sn = sanitize_name(str(rr["model"]))
                mlflow.log_metric(f"{sn}_latest_rmse", float(rr["latest_RMSE"]))
                mlflow.log_metric(f"{sn}_avg_rmse", float(rr["avg_RMSE"]))
        for p in [summary_path, latest_vs_avg_path, comparison_plot_path, all_metrics_path, all_preds_path]:
            if p and Path(p).exists():
                mlflow.log_artifact(str(p), artifact_path="comparison")

    print("MLflow logging complete")
else:
    print("MLflow skipped (disabled or package missing)")








